# 1. Import libraries

In [10]:
import pandas as pd
import numpy as np
import os
import time
from joblib import dump, load

from sklearn.model_selection import StratifiedKFold, cross_val_predict, RandomizedSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, make_scorer, precision_score, recall_score, f1_score, accuracy_score
from sklearn.decomposition import PCA

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline


# 2. Load data

In [11]:
data = pd.read_excel('złączone_dane.xlsx')
data = data.drop('image_id', axis=1)
data = data.drop(columns=[col for col in data.columns if any(x in col for x in ['3_p', '4_p', '5_p'])])


# 3. Preprocessing

In [12]:
X = data.drop('label', axis=1)
y = data['label']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

def apply_grouped_pca(X, n_components=1):
    lm_0_cols = [col for col in X.columns if col.startswith('0_point_lm_')]
    lm_1_cols = [col for col in X.columns if col.startswith('1_point_lm_')]
    lm_2_cols = [col for col in X.columns if col.startswith('2_point_lm_')]
    vec_cols = [col for col in X.columns if '_vec_' in col]

    def pca_transform(cols, prefix):
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X[cols])
        pca = PCA(n_components=n_components)
        X_pca = pca.fit_transform(X_scaled)
        return pd.DataFrame(X_pca, columns=[f'{prefix}_pca_{i}' for i in range(n_components)], index=X.index)

    pca_lm_0 = pca_transform(lm_0_cols, '0')
    pca_lm_1 = pca_transform(lm_1_cols, '1')
    pca_lm_2 = pca_transform(lm_2_cols, '2')
    vec_features = X[vec_cols].reset_index(drop=True)

    return pd.concat([pca_lm_0, pca_lm_1, pca_lm_2, vec_features], axis=1)

X_pca = apply_grouped_pca(X)


# 4. Trening MLP + RandomizedSearchCV + zapisywanie wyników

In [13]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    'accuracy': 'accuracy',
    'precision': make_scorer(precision_score, average='macro'),
    'recall': make_scorer(recall_score, average='macro'),
    'f1': make_scorer(f1_score, average='macro')
}

os.makedirs('models', exist_ok=True)
os.makedirs('logs', exist_ok=True)
os.makedirs('reports', exist_ok=True)

pipeline = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('classifier', MLPClassifier(random_state=42, max_iter=500))
])

param_grid = {
    'classifier__hidden_layer_sizes': [(50,), (100,), (50, 50)],
    'classifier__activation': ['relu', 'tanh'],
    'classifier__alpha': [0.0001, 0.001, 0.01],
    'classifier__learning_rate': ['constant', 'adaptive']
}

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=10,
    scoring='f1_macro',
    n_jobs=-1,
    cv=cv,
    verbose=1,
    random_state=42
)

print("🔍 Start treningu MLP...")
start_time = time.time()
search.fit(X_pca, y_encoded)
training_time = time.time() - start_time

best_model = search.best_estimator_

# Ewaluacja
y_pred = cross_val_predict(best_model, X_pca, y_encoded, cv=cv)
report = classification_report(y_encoded, y_pred, digits=4)

accuracy = accuracy_score(y_encoded, y_pred)
precision = precision_score(y_encoded, y_pred, average='macro')
recall = recall_score(y_encoded, y_pred, average='macro')
f1 = f1_score(y_encoded, y_pred, average='macro')

# Zapis
dump(best_model, 'models/mlp_best_model.pkl')
dump(le, 'models/mlp_label_encoder.pkl')

with open('reports/mlp_report.txt', 'w', encoding='utf-8') as f:
    f.write(f"Najlepszy model: MLPClassifier\n")
    f.write(f"Parametry: {search.best_params_}\n\n")
    f.write("=== Raport klasyfikacji ===\n")
    f.write(report)
    f.write("\n=== Metryki ogólne ===\n")
    f.write(f"Accuracy: {accuracy:.4f}\n")
    f.write(f"Precision (macro): {precision:.4f}\n")
    f.write(f"Recall (macro): {recall:.4f}\n")
    f.write(f"F1 Score (macro): {f1:.4f}\n")
    f.write(f"\nCzas treningu: {training_time:.2f} sekund\n")

with open('logs/mlp_log.txt', 'w', encoding='utf-8') as f:
    f.write(f"Najlepszy model: MLPClassifier\n")
    f.write(f"Parametry: {search.best_params_}\n")
    f.write(f"Czas treningu: {training_time:.2f} sekund\n")

print(report)


🔍 Start treningu MLP...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
              precision    recall  f1-score   support

           0     0.9867    0.9738    0.9802       305
           1     0.9962    1.0000    0.9981       526
           2     0.9934    0.9967    0.9951       303
           3     0.9464    0.8983    0.9217        59
           4     0.9949    1.0000    0.9974       388
           5     0.9942    0.9923    0.9933       521
           6     0.9962    0.9962    0.9962       530
           7     0.9912    0.9978    0.9945       450
           8     0.9459    0.8537    0.8974        41
           9     0.9865    1.0000    0.9932       439
          10     0.9983    1.0000    0.9991       573
          11     0.9983    1.0000    0.9992       599
          12     0.9943    0.9962    0.9953       529
          13     1.0000    0.9836    0.9917        61
          14     0.9956    0.9978    0.9967       452
          15     0.9962    1.0000    0.9981       

# 5. Ocena modelu po wczytaniu (z pliku) z KFold=5

In [14]:
# Wczytaj
model = load('models/mlp_best_model.pkl')
label_encoder = load('models/mlp_label_encoder.pkl')

# KFold ocena
y_pred = cross_val_predict(model, X_pca, y_encoded, cv=cv)
print("📊 Raport po cross-val (z pliku):")
print(classification_report(y_encoded, y_pred, digits=4))


📊 Raport po cross-val (z pliku):
              precision    recall  f1-score   support

           0     0.9867    0.9738    0.9802       305
           1     0.9962    1.0000    0.9981       526
           2     0.9934    0.9967    0.9951       303
           3     0.9464    0.8983    0.9217        59
           4     0.9949    1.0000    0.9974       388
           5     0.9942    0.9923    0.9933       521
           6     0.9962    0.9962    0.9962       530
           7     0.9912    0.9978    0.9945       450
           8     0.9459    0.8537    0.8974        41
           9     0.9865    1.0000    0.9932       439
          10     0.9983    1.0000    0.9991       573
          11     0.9983    1.0000    0.9992       599
          12     0.9943    0.9962    0.9953       529
          13     1.0000    0.9836    0.9917        61
          14     0.9956    0.9978    0.9967       452
          15     0.9962    1.0000    0.9981       525
          16     0.9912    0.9883    0.9898     

# 6. Przykład użycia modelu na nowych danych (test_data.xlsx)

In [15]:
import pandas as pd
import numpy as np
from joblib import load
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import os

# === ŚCIEŻKI ===
MODEL_PATH = 'models/mlp_best_model.pkl'  # <- zakładamy że model MLP był zapisany pod tą nazwą
ENCODER_PATH = 'models/mlp_label_encoder.pkl'
TEST_DATA_PATH = 'test_data.xlsx'

# === FUNKCJE POMOCNICZE ===
def preprocess_features(X: pd.DataFrame, n_components: int = 1) -> pd.DataFrame:
    """
    Przetwarza dane wejściowe:
    - osobne PCA dla każdej grupy punktów,
    - zachowuje cechy wektorowe,
    - skaluje dane przed PCA.
    """
    lm_0_cols = [col for col in X.columns if col.startswith('0_point_lm_')]
    lm_1_cols = [col for col in X.columns if col.startswith('1_point_lm_')]
    lm_2_cols = [col for col in X.columns if col.startswith('2_point_lm_')]
    vec_cols = [col for col in X.columns if '_vec_' in col]

    def pca_transform(cols, prefix):
        if not cols:
            return pd.DataFrame(index=X.index)
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X[cols])
        pca = PCA(n_components=min(n_components, len(cols)))
        X_pca = pca.fit_transform(X_scaled)
        return pd.DataFrame(X_pca, columns=[f'{prefix}_pca_{i}' for i in range(X_pca.shape[1])], index=X.index)

    pca_lm_0 = pca_transform(lm_0_cols, '0')
    pca_lm_1 = pca_transform(lm_1_cols, '1')
    pca_lm_2 = pca_transform(lm_2_cols, '2')
    vec_features = X[vec_cols].reset_index(drop=True)

    return pd.concat([pca_lm_0, pca_lm_1, pca_lm_2, vec_features], axis=1)

# === 1. Wczytaj model i encoder ===
if not os.path.exists(MODEL_PATH) or not os.path.exists(ENCODER_PATH):
    raise FileNotFoundError("Model lub encoder nie został znaleziony.")

model = load(MODEL_PATH)
label_encoder = load(ENCODER_PATH)

# === 2. Wczytaj dane testowe ===
df = pd.read_excel(TEST_DATA_PATH)

# Usuwanie niepotrzebnych kolumn (jak w treningu)
df = df.drop(columns=[col for col in df.columns if any(x in col for x in ['3_p', '4_p', '5_p'])], errors='ignore')
if 'image_id' in df.columns:
    df = df.drop('image_id', axis=1)

# === 3. Preprocessing ===
X_test = preprocess_features(df, n_components=1)

# === 4. Predykcja ===
y_pred_proba = model.predict_proba(X_test)

# === 5. Przykład: Prawdopodobieństwa dla pierwszej próbki ===
first_sample_proba = y_pred_proba[0]
class_labels = label_encoder.inverse_transform(np.arange(len(first_sample_proba)))
results = dict(zip(class_labels, np.round(first_sample_proba, 4)))

# === 6. Wynik ===
print('\n📊 Prawdopodobieństwa klas dla pierwszej próbki:')
for label, prob in results.items():
    print(f"{label}: {prob:.4f}")



📊 Prawdopodobieństwa klas dla pierwszej próbki:
a: 0.0007
a+: 0.9993
b: 0.0000
c: 0.0000
c+: 0.0000
ch: 0.0000
cz: 0.0000
d: 0.0000
e: 0.0000
e+: 0.0000
f: 0.0000
g: 0.0000
h: 0.0000
i: 0.0000
j: 0.0000
k: 0.0000
l: 0.0000
l+: 0.0000
m: 0.0000
n: 0.0000
n+: 0.0000
o: 0.0000
o+: 0.0000
p: 0.0000
r: 0.0000
rz: 0.0000
s: 0.0000
s+: 0.0000
sz: 0.0000
t: 0.0000
u: 0.0000
w: 0.0000
y: 0.0000
z: 0.0000
z+: 0.0000
z-: 0.0000


C:\Users\PC2\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\decomposition\_pca.py:586: RuntimeWarning: invalid value encountered in divide
  explained_variance_ = (S**2) / (n_samples - 1)
C:\Users\PC2\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\decomposition\_pca.py:586: RuntimeWarning: invalid value encountered in divide
  explained_variance_ = (S**2) / (n_samples - 1)
C:\Users\PC2\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\decomposition\_pca.py:586: RuntimeWarning: invalid value encountered in divide
  explained_variance_ = (S**2) / (n_samples - 1)
